In [1]:
import ipywidgets as widgets
from IPython.display import display, Audio
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import numpy as np
from dct_utils import *

In [2]:
sig, sr = librosa.load("samples/guitar-castanets.wav", sr=None, mono=True)
sig = sig / np.max(np.abs(sig))

# widgety
k_slider       = widgets.IntSlider(min=8, max=256, step=8, value=128,
                                    description='k (współczynniki)')
db_slider      = widgets.IntSlider(min=6, max=48, step=6, value=36,
                                    description='spreading dB')
frame_dropdown = widgets.Dropdown(options=[128, 256, 512, 1024],
                                   value=256, description='ramka')
output_audio   = widgets.Output()
output_plot    = widgets.Output()

def update(change=None):
    k         = k_slider.value
    db        = db_slider.value
    frame     = frame_dropdown.value
    rec       = compress_dct_psycho(sig, sr, frame, k, spreading_db=db)
    snr       = compute_snr(sig, rec)
    lsd       = log_spectral_distance(sig, rec, sr)

    with output_plot:
        output_plot.clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 3))
        fig.suptitle(f"SNR={snr:.1f}dB  LSD={lsd:.1f}  k={k}  db={db}  f={frame}")

        D_orig = librosa.amplitude_to_db(np.abs(librosa.stft(sig,   n_fft=512)), ref=np.max)
        D_rec  = librosa.amplitude_to_db(np.abs(librosa.stft(rec,   n_fft=512)), ref=np.max)

        librosa.display.specshow(D_orig, sr=sr, x_axis='time', y_axis='log', ax=axes[0])
        axes[0].set_title("Oryginał")

        librosa.display.specshow(D_rec,  sr=sr, x_axis='time', y_axis='log', ax=axes[1])
        axes[1].set_title("Skompresowany")

        # granice pasm Barksa
        bark_freqs = [100, 200, 300, 400, 510, 630, 770, 920, 1080, 1270,
                      1480, 1720, 2000, 2320, 2700, 3150, 3700, 4400, 5300,
                      6400, 7700, 9500, 12000, 15500]
        for f in bark_freqs:
            axes[1].axhline(f, color='white', linewidth=0.5, alpha=0.4)

        plt.tight_layout()
        plt.show()

    with output_audio:
        output_audio.clear_output(wait=True)
        display(Audio(rec, rate=sr, autoplay=False))

k_slider.observe(update,       names='value')
db_slider.observe(update,      names='value')
frame_dropdown.observe(update, names='value')

update()
display(widgets.VBox([
    widgets.HBox([k_slider, db_slider, frame_dropdown]),
    output_plot,
    output_audio
]))

In [5]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Audio

sig, sr = librosa.load("samples/577844__szegvari__summer-night-piano-solo.wav", sr=None, mono=True)
sig = sig / np.max(np.abs(sig))

frame_size = 256
output_plot  = widgets.Output()
output_audio = widgets.Output()

k_slider = widgets.IntSlider(min=8, max=256, step=8, value=128,
                              description='k', continuous_update=False)
db_slider = widgets.IntSlider(min=6, max=48, step=6, value=36,
                               description='spreading dB', continuous_update=False)

def hz_to_note(hz):
    if hz < 20 or np.isnan(hz):
        return None
    midi = 69 + 12 * np.log2(hz / 440)
    notes = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    return f"{notes[int(midi) % 12]}{int(midi) // 12 - 1}"

def get_piano_roll(signal, sr, fmin=80, fmax=2000):
    f0, voiced_flag, _ = librosa.pyin(signal, fmin=fmin, fmax=fmax, sr=sr)
    times = librosa.times_like(f0, sr=sr)
    return times, f0, voiced_flag

def update(change=None):
    k  = k_slider.value
    db = db_slider.value

    rec = compress_dct_psycho(sig, sr, frame_size, k, spreading_db=db)
    n   = min(len(sig), len(rec))
    snr = compute_snr(sig, rec)

    times_orig, f0_orig, voiced_orig = get_piano_roll(sig[:n],    sr)
    times_rec,  f0_rec,  voiced_rec  = get_piano_roll(rec[:n], sr)

    with output_plot:
        output_plot.clear_output(wait=True)
        fig, axes = plt.subplots(3, 1, figsize=(14, 10))
        fig.suptitle(f"Piano roll — k={k}  db={db}  SNR={snr:.1f}dB", fontsize=12)

        # --- oryginał piano roll ---
        ax = axes[0]
        ax.set_facecolor('#1a1a2e')
        for i in range(len(times_orig) - 1):
            if voiced_orig[i] and f0_orig[i] is not None and not np.isnan(f0_orig[i]):
                ax.barh(f0_orig[i], times_orig[i+1] - times_orig[i],
                        left=times_orig[i], height=f0_orig[i] * 0.04,
                        color='#00d4ff', alpha=0.85)
        ax.set_yscale('log')
        ax.set_ylim(80, 2000)
        ax.set_ylim(80, 2000)
        ax.set_ylabel("Częstotliwość [Hz]")
        ax.set_title("Oryginał")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}Hz"))
        ax.grid(True, alpha=0.2, color='white')

        # --- skompresowany piano roll ---
        ax = axes[1]
        ax.set_facecolor('#1a1a2e')

        orig_voiced_set = set(
            round(f, 1) for f, v in zip(f0_orig, voiced_orig)
            if v and f is not None and not np.isnan(f)
        )

        for i in range(len(times_rec) - 1):
            if voiced_rec[i] and f0_rec[i] is not None and not np.isnan(f0_rec[i]):
                f = f0_rec[i]
                closest = min(orig_voiced_set, key=lambda x: abs(x - f)) if orig_voiced_set else f
                survived = abs(closest - f) < f * 0.05
                color = '#00d4ff' if survived else '#ff4757'
                ax.barh(f, times_rec[i+1] - times_rec[i],
                        left=times_rec[i], height=f * 0.04,
                        color=color, alpha=0.85)

        ax.set_yscale('log')
        ax.set_ylim(80, 2000)
        ax.set_ylabel("Częstotliwość [Hz]")
        ax.set_title("Skompresowany  (niebieski = zachowana nuta, czerwony = zniekształcona)")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}Hz"))
        ax.grid(True, alpha=0.2, color='white')

        # --- mapa znikających nut (różnica) ---
        ax = axes[2]
        ax.set_facecolor('#1a1a2e')

        for i in range(len(times_orig) - 1):
            if voiced_orig[i] and f0_orig[i] is not None and not np.isnan(f0_orig[i]):
                f_orig = f0_orig[i]
                t      = times_orig[i]

                rec_idx = np.argmin(np.abs(times_rec - t))
                if (voiced_rec[rec_idx] and
                    f0_rec[rec_idx] is not None and
                    not np.isnan(f0_rec[rec_idx])):
                    f_rec = f0_rec[rec_idx]
                    err   = abs(f_orig - f_rec) / f_orig
                    color = plt.cm.RdYlGn(1 - min(err * 10, 1))
                else:
                    color = '#e74c3c'
                    err   = 1.0

                ax.barh(f_orig, times_orig[i+1] - times_orig[i],
                        left=times_orig[i], height=f_orig * 0.04,
                        color=color, alpha=0.9)

        ax.set_yscale('log')
        ax.set_ylim(80, 2000)
        ax.set_xlabel("Czas [s]")
        ax.set_ylabel("Częstotliwość [Hz]")
        ax.set_title("Błąd pitch (zielony = zachowany, czerwony = zgubiony)")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}Hz"))
        ax.grid(True, alpha=0.2, color='white')

        plt.tight_layout()
        plt.show()

    with output_audio:
        output_audio.clear_output(wait=True)
        display(Audio(rec, rate=sr, autoplay=False))

k_slider.observe(update,  names='value')
db_slider.observe(update, names='value')

update()
display(widgets.VBox([
    widgets.HBox([k_slider, db_slider]),
    output_plot,
    output_audio,
]))